# Monte Carlo不确定性传播

这个示例把几个常见风电评估环节写成随机输入，观察它们如何共同形成Net AEP代理分布。

示例中的不确定度分布和参数范围由代码人为设定，只用于说明传播机制，不对应具体项目，也不代表行业标准取值。

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)
n_mc = 20_000

# 构造一个长期风速样本，作为基础风况。
n_wind = 50_000
shape_k = 2.2
target_mean = 7.5
scale_c = target_mean / math.gamma(1.0 + 1.0 / shape_k)
base_ws = scale_c * rng.weibull(shape_k, n_wind)

基础风况保持固定，每一次Monte Carlo抽样只改变下列不确定输入：

- 风速尺度修正因子；
- 尾流损失；
- 可利用率损失；
- 电气损失。

这样可以看到上游风速不确定性和下游损失不确定性如何共同影响最终结果。

In [ ]:
def cf_proxy(ws):
    ws = np.asarray(ws, dtype=float)
    out = np.zeros_like(ws)
    partial = (ws >= 3.0) & (ws < 12.0)
    rated = (ws >= 12.0) & (ws <= 25.0)
    out[partial] = ((ws[partial] - 3.0) / 9.0) ** 3
    out[rated] = 1.0
    return out

wind_scale = rng.normal(loc=1.0, scale=0.03, size=n_mc)

wake_loss = np.clip(
    rng.normal(loc=0.08, scale=0.02, size=n_mc),
    0.0, 0.30
)

availability_loss = np.clip(
    rng.normal(loc=0.03, scale=0.01, size=n_mc),
    0.0, 0.15
)

electrical_loss = np.clip(
    rng.normal(loc=0.02, scale=0.005, size=n_mc),
    0.0, 0.10
)

为了避免在每次抽样中重复计算完整风速序列，先建立“风速尺度因子→平均CF代理”的查找关系，再进行插值。这个处理只服务于计算效率。

In [ ]:
scale_grid = np.linspace(0.88, 1.12, 301)
cf_grid = np.array([
    cf_proxy(base_ws * s).mean()
    for s in scale_grid
])

gross_cf = np.interp(wind_scale, scale_grid, cf_grid)

net_cf = (
    gross_cf
    * (1.0 - wake_loss)
    * (1.0 - availability_loss)
    * (1.0 - electrical_loss)
)

aep_proxy = net_cf * 8760.0

p50 = np.percentile(aep_proxy, 50)
p90 = np.percentile(aep_proxy, 10)  # 90% exceedance probability

print(f"mean Net AEP proxy = {aep_proxy.mean():.1f} full-load-hours")
print(f"P50 = {p50:.1f} full-load-hours")
print(f"P90 = {p90:.1f} full-load-hours")
print(f"P50 - P90 = {p50 - p90:.1f} full-load-hours")

采用“超越概率”定义时，P90表示有90%概率被超过的AEP，对应经验分布的10%分位点。

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist(aep_proxy, bins=70, density=True, alpha=0.75)
plt.axvline(p50, linestyle="--", label="P50")
plt.axvline(p90, linestyle="--", label="P90")
plt.xlabel("Net AEP proxy (full-load-hours)")
plt.ylabel("Probability density")
plt.legend()
plt.tight_layout()
plt.show()

## 单因素敏感度

分别固定其他输入，只改变一个变量，用标准差粗略比较各输入对结果波动的贡献。该结果仅用于敏感度检查，不等同于严格的方差分解。

In [ ]:
def std_from_one_variable(kind):
    if kind == "wind":
        gross = np.interp(wind_scale, scale_grid, cf_grid)
        out = gross * (1 - 0.08) * (1 - 0.03) * (1 - 0.02)
    elif kind == "wake":
        gross = np.interp(np.ones(n_mc), scale_grid, cf_grid)
        out = gross * (1 - wake_loss) * (1 - 0.03) * (1 - 0.02)
    elif kind == "availability":
        gross = np.interp(np.ones(n_mc), scale_grid, cf_grid)
        out = gross * (1 - 0.08) * (1 - availability_loss) * (1 - 0.02)
    elif kind == "electrical":
        gross = np.interp(np.ones(n_mc), scale_grid, cf_grid)
        out = gross * (1 - 0.08) * (1 - 0.03) * (1 - electrical_loss)
    return np.std(out * 8760.0)

for name in ["wind", "wake", "availability", "electrical"]:
    print(f"{name:>12}: std={std_from_one_variable(name):.1f} full-load-hours")

风速进入非线性功率曲线，上游尺度误差可能被放大。损失项则以乘法形式进入Net AEP。

真实项目还需要处理输入分布是否合理、不同不确定源之间的相关性、年际变率与知识不确定性的区分，以及预测分布的后验校验。

参考：

- Lee & Fields (2021), https://doi.org/10.5194/wes-6-311-2021
- Barber et al. (2022), https://doi.org/10.5194/wes-7-1503-2022
- Hammond & Simley (2026), https://doi.org/10.5194/wes-11-1251-2026